In [ ]:
import os
import librosa
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

# PATHS
BASE_DIR = os.getcwd()
# Ensure this matches the 'data' folder in your Git repo
RAW_DATA_PATH = os.path.join(BASE_DIR, "data", "genres_original")
FEAT_DIR = os.path.join(BASE_DIR, "features")

os.makedirs(FEAT_DIR, exist_ok=True)
# SETTINGS
SR = 22050
CLIP_SEC = 3
N_MFCC = 20
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 
          'jazz', 'metal', 'pop', 'reggae', 'rock']

In [12]:
def extract_ventral_features(segment, sr):
    """Statistical snapshots for XGBoost"""
    # 1. Spectral Centroid
    sc = librosa.feature.spectral_centroid(y=segment, sr=sr)
    # 2. Chroma
    chroma = librosa.feature.chroma_stft(y=segment, sr=sr)
    # 3. RMS (Loudness)
    rms = librosa.feature.rms(y=segment)
    
    return {
        'centroid_mean': np.mean(sc),
        'centroid_var': np.var(sc),
        'chroma_mean': np.mean(chroma),
        'chroma_var': np.var(chroma),
        'rms_mean': np.mean(rms),
        'rms_var': np.var(rms)
    }

def extract_dorsal_features(segment, sr):
    """Sequential MFCCs for LSTM"""
    mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=N_MFCC)
    return mfcc.T # Shape (Time_Steps, 20)

In [13]:
stats_data = []
temporal_data = []
labels = []
samples_per_clip = SR * CLIP_SEC

for genre in GENRES:
    folder = os.path.join(RAW_DATA_PATH, genre)
    files = sorted([f for f in os.listdir(folder) if f.endswith('.wav')])
    
    for fname in tqdm(files, desc=f"Processing {genre}"):
        path = os.path.join(folder, fname)
        try:
            y, _ = librosa.load(path, sr=SR)
            # Slicing 30s into 3s chunks
            for start in range(0, len(y) - samples_per_clip + 1, samples_per_clip):
                seg = y[start : start + samples_per_clip]
                
                # Stream 1
                stats_data.append(extract_ventral_features(seg, SR))
                # Stream 2
                temporal_data.append(extract_dorsal_features(seg, SR))
                # Label
                labels.append(genre)
        except:
            continue

# --- SAVING EVERYTHING ---
# 1. Labels
le = LabelEncoder()
y_encoded = le.fit_transform(labels)
np.save(os.path.join(FEAT_DIR, 'y_labels.npy'), y_encoded)
np.save(os.path.join(FEAT_DIR, 'label_names.npy'), le.classes_)

# 2. XGBoost Data
df = pd.DataFrame(stats_data)
df['label'] = y_encoded
df.to_csv(os.path.join(FEAT_DIR, 'statistical_features.csv'), index=False)

# 3. LSTM Data
np.save(os.path.join(FEAT_DIR, 'temporal_sequences.npy'), np.array(temporal_data))

print(f"✅ Success! Your 'features/' folder now contains {len(os.listdir(FEAT_DIR))} files.")


/home/ulas/Uni/Neuroinformatik/Music_Classification_Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Exception ignored in: <function tqdm.__del__ at 0x7a3218ec7240>
Traceback (most recent call last):
  File "/home/ulas/Uni/Neuroinformatik/Music_Classification_Project/.venv/lib/python3.12/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/ulas/Uni/Neuroinformatik/Music_Classification_Project/.venv/lib/python3.12/site-packages/tqdm/notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'




































































































Processing jazz:  54%|█████▍    | 54/100 [00:15<00:13,  3.39it/s]/tmp/ipy

✅ Success! Your 'features/' folder now contains 4 files.
